# Building Your First AI Chatbot

Welcome! In this lab, you'll build your own AI chatbot.

## About This Environment

You're running this notebook on an **AMD MI300 GPU** in the **AMD Developer Cloud**. The MI300 is a powerful GPU with 192GB of high-bandwidth memory, making it perfect for running large language models quickly and efficiently.

Throughout this tutorial, when you chat with your AI bot, the inference is running on AMD hardware!

## What You'll Learn

1. [What is inference?](#what-is-inference)
2. [Loading the AI model](#loading-model)
3. [Building a basic chatbot](#basic-chatbot)
4. [Customizing with system prompts](#custom-chatbot)
5. [Controlling AI behavior with parameters](#parameters)
6. [Adding a web interface](#widget-gui)

---

<a id="what-is-inference"></a>

## What is Inference?

If you've chatted with AI bots like ChatGPT or Claude, you've already used **inference**!

**Inference** is when you send an input to a trained generative AI model and it generates an output (e.g. text, code, image).

In this tutorial, we'll use **vLLM** to host and run a model directly on AMD's MI300 GPU.

---

<a id="loading-model"></a>

## Loading the AI Model

Before we can chat, we need to load a pre-trained model onto the MI300 GPU. We'll use **Qwen 3**, a popular open-source language model with over 3 million downloads.

Run the cell below. Note that it may take a minute or two to load the model into GPU memory.

In [1]:
from vllm import LLM, SamplingParams

# Load the AI model on the MI300 GPU
# This takes about a minute - you'll see loading messages
model_name = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading {model_name} on AMD MI300 GPU...")
llm = LLM(model=model_name)
print("✓ Model loaded and ready!")

Loading Qwen/Qwen3-4B-Instruct-2507 on AMD MI300 GPU...
INFO 10-31 06:02:46 [api_utils.py:273] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}


INFO 10-31 06:02:53 [model.py:623] Resolved architecture: Qwen3ForCausalLM
INFO 10-31 06:02:53 [model.py:1788] Using max model len 262144
INFO 10-31 06:02:53 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 10-31 06:02:53 [vllm.py:1109] Asynchronous scheduling is enabled.
INFO 10-31 06:02:53 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=891) INFO 10-31 06:02:55 [core.py:116] Initializing a V1 LLM engine (v0.26.0) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs,

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=891) INFO 10-31 06:03:01 [default_loader.py:430] Loading weights took 4.03 seconds
(EngineCore pid=891) INFO 10-31 06:03:02 [model_runner.py:305] Model loading took 7.61 GiB and 5.553090 seconds
(EngineCore pid=891) INFO 10-31 06:03:08 [backends.py:1094] Using cache directory: /root/.cache/vllm/torch_compile_cache/f6591386d8/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=891) INFO 10-31 06:03:08 [backends.py:1155] Dynamo bytecode transform time: 5.76 s
(EngineCore pid=891) INFO 10-31 06:03:14 [backends.py:378] Cache the graph of compile range (1, 16384) for later use
(EngineCore pid=891) INFO 10-31 06:03:18 [backends.py:393] Compiling a graph for compile range (1, 16384) takes 9.45 s
(EngineCore pid=891) INFO 10-31 06:03:21 [decorators.py:708] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/da516c7dcf2c18f0c21113afe276120bba56112e2606ccee1ba735b7d819d164/rank_0_0/model
(EngineCore pid=891) INFO 10-31 06:03:21 [monitor.p

Capturing CUDA graphs (FULL):   0%|          | 0/51 [00:00<?, ?it/s]

(EngineCore pid=891) WARNING 10-31 06:03:26 [chunked_prefill_paged_decode.py:419] Cannot use ROCm custom paged attention kernel, falling back to Triton implementation.


Capturing CUDA graphs (FULL): 100%|██████████| 51/51 [00:02<00:00, 22.63it/s]


(EngineCore pid=891) INFO 10-31 06:03:28 [model_runner.py:747] Graph capturing finished in 5 secs, took 4.10 GiB
(EngineCore pid=891) INFO 10-31 06:03:28 [gpu_worker.py:857] Free memory on device (191.18/191.69 GiB) on startup. Desired GPU memory utilization is (0.92, 176.35 GiB). Actual usage is 7.61 GiB for weight, 1.25 GiB for peak activation, 0.43 GiB for non-torch memory, and 4.1 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=174826960425` (162.82 GiB) to fit into requested memory, or `--kv-cache-memory=190751726080` (177.65 GiB) to fully utilize gpu memory. Current kv cache memory in use is 167.07 GiB.
(EngineCore pid=891) INFO 10-31 06:03:37 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=891) INFO 10-31 06:03:38 [core.py:340] init engine (profile, create kv cache, warmup model) took 35.76 s (compilation: 18.77 s)
(EngineCore pid=891) INFO 10-31 06:03:38 [ke

<a id="basic-chatbot"></a>

## Building a Basic Chatbot

Great! Now that the model is loaded, let's build a simple chatbot.

The code below creates a conversation loop where you can chat with your AI bot. After you run the cell, you can type your messages and press Enter to converse. Give it a try! When you're ready to continue, type `exit` or `quit` to end the chat.

In [2]:
# Store our conversation history
conversation = []

# Set a reasonable response length
sampling_params = SamplingParams(max_tokens=200)

# Chat loop
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    
    # Add user message to conversation
    conversation.append({"role": "user", "content": user_input})
    
    # This is INFERENCE - send to the model and get a response!
    outputs = llm.chat(conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text
    
    # Add bot's response to conversation so it remembers context
    conversation.append({"role": "assistant", "content": bot_message})
    
    print(f"Bot: {bot_message}\n")

You:  exit


Goodbye!


### What Just Happened?

Congratulations, you just performed **inference**!

Here's what happened each time you sent a message:

1. Your text was added to the conversation history
2. The conversation was sent to the Qwen 3 model running on the **MI300 GPU**
3. The GPU processed billions of calculations in parallel
4. The model generated a response based on patterns learned during training
5. The response came back to you in seconds

This is how ChatGPT, Claude, and other AI chatbots work!

**Note:** The chatbot remembers your conversation because we save both your messages and its responses in the `conversation` list.

---

<a id="custom-chatbot"></a>

## Customizing Your Chatbot with System Prompts

Want to give your chatbot a personality? That's where **system prompts** come in!

A **system prompt** is a special instruction that tells the AI how to behave. It's like giving the chatbot a role to play.

Let's create a Shakespeare chatbot that speaks in Elizabethan English. Run the cell and try a few messages! When you're ready to continue, type `exit` or `quit` to end the chat.

In [3]:
# Create a conversation with a system prompt
shakespeare_conversation = [
    {
        "role": "system",  # This is the special system message
        "content": "You are a chatbot that speaks like William Shakespeare. Use Elizabethan English and poetic language. Be helpful but stay in character!"
    }
]

# Set a reasonable response length
sampling_params = SamplingParams(max_tokens=200)

# Chat loop
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Farewell!")
        break
    
    shakespeare_conversation.append({"role": "user", "content": user_input})
    
    outputs = llm.chat(shakespeare_conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text
    
    shakespeare_conversation.append({"role": "assistant", "content": bot_message})
    
    print(f"Shakespeare Bot: {bot_message}\n")

You:  exit


Farewell!


### How System Prompts Work

Notice how the chatbot's personality completely changed? That's the power of system prompts!

The conversation list starts with a message that has:
- `"role": "system"` - tells the AI this is an instruction, not a user message
- `"content": "..."` - the actual instruction about how to behave

This is how bots are polite, how coding assistants know to write code, and how different AI tools have different personalities.

### Try It Yourself!

Create your own custom chatbot! Some ideas:
- **Pirate Bot**: "You are a friendly pirate. Talk like a pirate!"
- **Rap Bot**: "You are a talented rapper. Respond only in rhymes and rap lyrics."
- **Cat Bot**: "You are a cat. Respond with meows."
- **Science Bot**: "You are a scientist who explains things with facts and examples."

Modify the system prompt below to see how the model responds differently to "Hello! Tell me about yourself".

In [4]:
# Create your own custom chatbot!
my_conversation = [
    {
        "role": "system",
        "content": "You are a friendly pirate. Talk like a pirate!"  # <-- Change this!
    }
]

# Test it with a quick message
my_conversation.append({"role": "user", "content": "Hello! Tell me about yourself."})

sampling_params = SamplingParams(max_tokens=200)
outputs = llm.chat(my_conversation, sampling_params)

print(outputs[0].outputs[0].text)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 10-31 06:04:16 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s, est. speed input: 42.54 toks/s, output: 274.47 toks/s]

Arrr! Ye’ve found the right cove, matey! I’m a pint-sized pirate with a brain full o’ nautical nooks and crannies—y’know, I’m here to help ye with any quest ye might be on! Whether it's solving a riddle, cracking a code, or just needin’ a lil’ piece o’ advice—be it sunny, stormy, or just a bit barnacled—I’ll be yer trusty compass (and possibly a snack o’ sugar-cane rum!).

I spout answers in the most roguish of manner—swashbucklin’ with a cheeky tone, full of spice and a good ol’ rapscallion charm. I don’t just answer questions—I *relate* to ‘em, like a sea-faring comrade who's sailed past the tropics and seen the ghost ships!

So now that ye’ve met me, what’s yer quest, ye wily


<a id="parameters"></a>

## Controlling AI Behavior with Parameters

You've already been using `SamplingParams` with `max_tokens` to control response length. But there are other parameters that control **how** the AI generates text:

### The Three Key Parameters

1. **temperature** - Controls creativity and randomness
2. **max_tokens** - Controls response length (you've been using this!)
3. **top_p** - Controls word choice diversity

Let's explore how temperature affects responses!

In [5]:
# Let's compare LOW vs HIGH temperature
question = "Tell me a fun fact about cats."

print("🔵 LOW Temperature (0.2) - More predictable and focused:\n")
sampling_params = SamplingParams(temperature=0.2, max_tokens=200)
outputs = llm.chat([{"role": "user", "content": question}], sampling_params)
print(outputs[0].outputs[0].text)

print("\n" + "="*60 + "\n")

print("🔴 HIGH Temperature (0.9) - More creative and varied:\n")
sampling_params = SamplingParams(temperature=0.9, max_tokens=200)
outputs = llm.chat([{"role": "user", "content": question}], sampling_params)
print(outputs[0].outputs[0].text)

print("\n" + "="*60)
print("💡 Notice the difference? Higher temperature = more variety!")

🔵 LOW Temperature (0.2) - More predictable and focused:



Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s, est. speed input: 35.84 toks/s, output: 286.68 toks/s]

Sure! Here's a fun fact: Cats have a unique ability to land on their feet when they fall — a skill called the "righting reflex." This reflex kicks in as soon as they start to fall, allowing them to rotate their bodies mid-air and orient themselves so they land on their feet. Scientists believe this ability is due to a combination of their flexible spine, inner ear balance, and the way they tuck their legs and twist their bodies. It’s like nature’s built-in landing pad — and it’s one reason why cats are so graceful, even when they’re falling from a high shelf! 🐾✨


🔴 HIGH Temperature (0.9) - More creative and varied:



Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s, est. speed input: 35.33 toks/s, output: 287.05 toks/s]

Sure! Here's a fun fact: Cats have a unique ability to land on their feet when they fall — a skill called the "righting reflex." This reflex kicks in almost instantly, allowing them to twist their bodies mid-air and reorient themselves so they land on their feet. Scientists believe this happens because cats can sense the direction of gravity through their inner ears, and they use a combination of body rotation and muscle control to flip themselves. 

Bonus fun twist: This reflex actually works best when they fall from a height of about 2 to 3 stories or less — beyond that, they might not survive to land safely! 😸

💡 Notice the difference? Higher temperature = more variety!


### Parameter Guide

Here's a quick reference for when to use different parameter values:

**temperature** (0.0 to 1.0)
- **0.1-0.3** → Predictable, consistent, factual (good for Q&A, facts)
- **0.5-0.7** → Balanced (good for general chatting)
- **0.8-1.0** → Creative, varied, surprising (good for storytelling, brainstorming)

**max_tokens**
- **50-100** → Short, concise answers
- **150-300** → Medium responses (good default)
- **500+** → Long, detailed explanations

**top_p** (0.0 to 1.0)
- **0.1-0.5** → Conservative word choices
- **0.7-0.9** → Balanced (good default is 0.9)
- **0.95-1.0** → Maximum diversity

Now try experimenting yourself!

In [8]:
# Experiment with different parameter combinations!
# Try changing these values and see how the response changes

sampling_params = SamplingParams(
    temperature=0.2,    # Try: 0.2 (safe), 0.7 (balanced), or 0.9 (creative)
    max_tokens=200,     # Try: 50 (short), 200 (medium), or 300 (long)
    top_p=0.9          # Try: 0.5 (more focused), 0.9 (balanced), or 1.0 (more diverse)
)

outputs = llm.chat(
    [{"role": "user", "content": "What's the best way to learn AI?"}],
    sampling_params
)

print(outputs[0].outputs[0].text)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s, est. speed input: 22.75 toks/s, output: 267.69 toks/s]

There's no single "best" way to learn AI, as the ideal approach depends on your goals, background, and available time. However, here’s a structured, practical, and effective roadmap to help you learn AI efficiently and meaningfully:

---

### 🚀 Step-by-Step Guide to Learning AI

#### 1. **Define Your Goal**
Ask yourself:
- Are you interested in **building AI models** (e.g., machine learning, deep learning)?
- Wanting to **apply AI** in a specific field (e.g., healthcare, finance, gaming)?
- Interested in **AI ethics, policy, or business applications**?
- Or just curious about how AI works?

👉 *Example:* If you're a developer, focus on building models. If you're a business person, focus on AI applications and impact.

---

#### 2. **Build a Strong Foundation (1–2 Months)**

**Core Topics to Learn:**
- **Mathematics**: Linear algebra


<a id="widget-gui"></a>

## Adding a Graphical Interface

Want a real chat interface? We can use **ipywidgets** to create a quick interactive UI with sliders! 

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create widgets
message_box = widgets.Textarea(
    placeholder='Type your message here...',
    description='Message:',
    layout=widgets.Layout(width='100%', height='80px')
)

temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Temperature:',
    style={'description_width': '150px'}
)

max_tokens_slider = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=10,
    description='Max Tokens:',
    style={'description_width': '150px'}
)

top_p_slider = widgets.FloatSlider(
    value=0.9,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Top P:',
    style={'description_width': '150px'}
)

send_button = widgets.Button(
    description='Send',
    button_style='success',
    icon='paper-plane'
)

output_area = widgets.Output()

# Chat function
def on_send_clicked(b):
    with output_area:
        clear_output()
        message = message_box.value
        if message.strip():
            print(f"You: {message}\n")
            
            sampling_params = SamplingParams(
                temperature=temperature_slider.value,
                max_tokens=int(max_tokens_slider.value),
                top_p=top_p_slider.value
            )
            
            outputs = llm.chat([{"role": "user", "content": message}], sampling_params)
            response = outputs[0].outputs[0].text
            
            print(f"Bot: {response}")

send_button.on_click(on_send_clicked)

# Display the interface
display(widgets.HTML("<h3>AI Chatbot on AMD MI300</h3>"))
display(widgets.HTML("<p>Chat with AI running on AMD hardware!</p>"))
display(message_box)
display(temperature_slider)
display(max_tokens_slider)
display(top_p_slider)
display(send_button)
display(output_area)

HTML(value='<h3>AI Chatbot on AMD MI300</h3>')

HTML(value='<p>Chat with AI running on AMD hardware!</p>')

Textarea(value='', description='Message:', layout=Layout(height='80px', width='100%'), placeholder='Type your …

FloatSlider(value=0.7, description='Temperature:', max=1.0, min=0.1, style=SliderStyle(description_width='150p…

IntSlider(value=200, description='Max Tokens:', max=500, min=50, step=10, style=SliderStyle(description_width=…

FloatSlider(value=0.9, description='Top P:', max=1.0, min=0.1, style=SliderStyle(description_width='150px'))

Button(button_style='success', description='Send', icon='paper-plane', style=ButtonStyle())

Output()

---

## Summary: What You've Learned

Congratulations! You've learned the fundamentals of AI inference on AMD hardware. 🎉

### Key Concepts

**1. Inference**
- Inference means using a trained AI model to generate outputs
- Every time you sent a message to the chatbot, you performed inference
- The AMD MI300 GPU made this fast by doing billions of calculations in parallel

**2. System Prompts**  
- System prompts tell the AI how to behave
- They give the chatbot a personality or role
- This is how AI tools can get specific behaviors

**3. Parameters**
- `temperature` → Controls creativity
- `max_tokens` → Controls response length
- `top_p` → Controls word diversity

### The Core Code Pattern

Here's the simple pattern you learned:

```python
# 1. Load model (once)
llm = LLM(model="Qwen/Qwen3-4B-Instruct-2507")

# 2. Create conversation with optional system prompt
conversation = [
    {"role": "system", "content": "Your instructions here"},
    {"role": "user", "content": "Your message"}
]

# 3. Do inference
sampling_params = SamplingParams(temperature=0.7, max_tokens=200)
outputs = llm.chat(conversation, sampling_params)
response = outputs[0].outputs[0].text
```

That's the foundation of generative AI!

### AI on AMD

You experienced real AI running on:
- ✅ **AMD MI300 GPU** - Fast, powerful GPU with 192GB of high-bandwidth memory
- ✅ **vLLM** - Optimized inference engine for AMD ROCm
- ✅ **AMD Developer Cloud** - Cloud access to MI300 GPU

---

### Keep Experimenting!

Try these challenges:
- Make a chatbot that speaks a different language
- Create a chatbot that rhymes every response
- Build a quiz bot that asks questions
- Experiment with extreme temperature values (0.1 vs 1.0)

And see how easy it can be to build from — like making a quick GUI with widgets, adding conversation history, or creating specialized assistants for different tasks!